# 20 · LLM batch runs
Runs `run_inference.py` for every (model, task, split) job in `CONFIG`, one subprocess per job: each job loads its model fresh, so GPU memory is freed and one crash doesn't stop the batch.
- **Inputs:** `configs/llms.yaml` (15 models, pinned revisions), `data/processed/{dev,test}` (test is label-blind: no label columns, no metrics).
- **Outputs:** one `OUTPUT_ROOT/{run_name}/` per job (see `run_inference.py`), plus this batch's folder with `config.json`, a status table per session and one log per job.
- **Resume:** run names are fixed by `CONFIG["date"]`, so rerunning the notebook skips finished jobs and resumes partial ones. Keep the date the same across Colab sessions.
- `dry_run: True` (default) only prints the job table. Set it to `False` on a GPU runtime to launch.

In [1]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Sexism-research")  # TODO: set to your Drive clone
else:
    PROJECT_ROOT = Path.cwd().parent  # local run from notebooks/
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"  # on Drive, so runs survive disconnects
# On Colab, weights go to local disk: Drive is slow for multi-GB reads and 15 models need ~150 GB
HF_HOME = Path("/content/hf_cache") if IN_COLAB else PROJECT_ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)
sys.path.insert(0, str(PROJECT_ROOT))

# Copy the processed parquet files to local disk for speed; outputs still go to Drive
if IN_COLAB:
    LOCAL_DATA = Path("/content/data")
    shutil.copytree(DATA_ROOT / "processed", LOCAL_DATA / "processed", dirs_exist_ok=True)
    DATA_ROOT = LOCAL_DATA
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT  ", DATA_ROOT, "\nOUTPUT_ROOT", OUTPUT_ROOT, "\nHF_HOME    ", HF_HOME)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Sexism-research/data/processed'

In [2]:
!ls 

drive  sample_data


In [ ]:
# Same pins as 00 and requirements.txt; torch is Colab's build, logged not pinned
PINS = {
    "pandas": "3.0.6", "numpy": "2.5.4", "scikit-learn": "1.9.1", "pyarrow": "26.0.0", "pyyaml": "6.0.3",
    "transformers": "5.20.1", "pydantic": "2.13.5", "outlines": "1.3.3", "xgrammar": "0.2.8",
}
if IN_COLAB:
    PIP_ARGS = " ".join(f"{p}=={v}" for p, v in PINS.items())
    %pip install -q {PIP_ARGS}
    print("If pip changed anything, restart the runtime, then run all again")

from importlib.metadata import version

off_pin = {p: version(p) for p in PINS if version(p) != PINS[p]}
assert not off_pin, f"installed {off_pin} differ from PINS: install the pins and restart the kernel"

In [ ]:
import datetime

# The only cell to edit between batches
CONFIG = {
    "date": "20261020",  # fixed batch date: part of every run name, keep it to resume
    "models": "all",  # "all" = every model in configs/llms.yaml, or a list of short names
    "tasks": [
        ("cmsb", "binary"),
        ("edos", "task_a"), ("edos", "task_b"), ("edos", "task_c"),
        ("exist", "task_1_1"), ("exist", "task_1_2"), ("exist", "task_1_3"),
    ],
    "splits": ["dev", "test"],  # dev first: thresholds and the refusal cutoff are frozen on dev
    "backend": "logprob",
    "limit": None,  # e.g. 5 for a smoke batch (changes the run names)
    "seed": 42,
    # TODO: think-on runs (Qwen3.5-4B) need a generate-then-score path; JSON-constrained decoding blocks thinking
    "thinking_models": [],
    "dry_run": True,  # True: print the job table only
    "stop_on_error": False,
}
CONFIG["run_name"] = f"{CONFIG['date']}_all_all_llms_batch_s{CONFIG['seed']}"
print(CONFIG["run_name"])

In [ ]:
from src.utils import set_seeds

# Each job sets and logs its own seeds in run_inference.py; this covers the notebook process only
set_seeds(CONFIG["seed"])

In [ ]:
# Token from Colab secrets (or the local env), passed to the job processes, never printed
if IN_COLAB:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF_TOKEN set:", bool(HF_TOKEN), "(Llama 3.1 and Aya Expanse are gated and need it)")

import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.0f} GB")
else:
    print("no GPU: keep dry_run True here")

In [ ]:
from src.utils import build_jobs, load_llm_registry, status_table

REGISTRY = load_llm_registry(PROJECT_ROOT / "configs" / "llms.yaml")
MODELS = list(REGISTRY) if CONFIG["models"] == "all" else CONFIG["models"]
JOBS = build_jobs(
    REGISTRY, MODELS, CONFIG["tasks"], CONFIG["splits"],
    date=CONFIG["date"], seed=CONFIG["seed"], backend=CONFIG["backend"], limit=CONFIG["limit"],
    thinking_models=tuple(CONFIG["thinking_models"]),
)
BATCH_DIR = OUTPUT_ROOT / CONFIG["run_name"]
before = status_table(OUTPUT_ROOT, JOBS)
print(f"{len(JOBS)} jobs:", before.status.value_counts().to_dict())
before.pivot_table(index="model", columns="status", values="run_name", aggfunc="count", fill_value=0)

In [ ]:
import json

from src.utils import job_status, run_job, tail

BATCH_DIR.mkdir(parents=True, exist_ok=True)
(BATCH_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, default=str))
SCRIPT = PROJECT_ROOT / "run_inference.py"
ENV = {"DATA_ROOT": str(DATA_ROOT), "OUTPUT_ROOT": str(OUTPUT_ROOT), "HF_HOME": str(HF_HOME)}
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN

FAILED = {}
todo = [j for j in JOBS if job_status(OUTPUT_ROOT, j)[0] in ("todo", "partial")]
if CONFIG["dry_run"]:
    print(f"dry run: {len(todo)} jobs would run; set dry_run False to launch")
else:
    for k, job in enumerate(todo, 1):
        print(f"[{k}/{len(todo)}] {job.run_name}")
        code, log = run_job(job, SCRIPT, ENV, BATCH_DIR / "logs")
        if code != 0:
            FAILED[job.run_name] = tail(log)
            print("  FAILED:\n" + FAILED[job.run_name])
            if CONFIG["stop_on_error"]:
                break
print("failed:", list(FAILED) or "none")

In [ ]:
# Status record for this session; earlier sessions' tables are kept, never overwritten
stamp = f"{datetime.datetime.now():%Y%m%d-%H%M%S}"
after = status_table(OUTPUT_ROOT, JOBS)
after.to_csv(BATCH_DIR / f"status_{stamp}.csv", index=False)
(BATCH_DIR / f"failed_{stamp}.json").write_text(json.dumps(FAILED, indent=2))
counts = after.status.value_counts().to_dict()
(BATCH_DIR / "README.md").write_text(f"""# {CONFIG['run_name']}
- Purpose: batch LLM inference (`notebooks/20_llm_runs.ipynb`), one `run_inference.py` subprocess per job.
- Config: `config.json` ({len(MODELS)} models, {len(CONFIG['tasks'])} tasks, splits {', '.join(CONFIG['splits'])}, backend {CONFIG['backend']}).
- Latest session {stamp}: {counts}; per-session tables in `status_*.csv`, failures in `failed_*.json`, logs in `logs/`.
- Metrics live in each dev run's own folder; test runs are label-blind and scored in `99_final_eval`.
""")
print(counts)
after[after.status != "done"]

## Summary
TODO after the first full batch: number of finished / failed / blocked jobs per model, refusal-mass observations on dev, and anything that needed a rerun. Results and per-run metrics are in each run folder; this notebook only orchestrates.